# Časť 3, Téma 3: DPA na firmvérovú implementáciu AES

---
POZNÁMKA: Toto laboratórne cvičenie odkazuje na niektoré (komerčné) školiace materiály na [ChipWhisperer.io](https://www.ChipWhisperer.io). Lab môžete voľne spúšťať a používať v súlade s open-source licenciou (vrátane jeho použitia vo vašich vlastných kurzoch, ak ich šírite podobným spôsobom), ale musíte zachovať upozornenie o tomto pôvodnom zdroji. Zvážte účasť v našom školiacom kurze, aby ste si odniesli plnohodnotný zážitok.

---

**ZHRNUTIE:** *V predchádzajúcom labe ste videli, ako sa dá použiť jediný bit informácie na obnovenie celého bajtu kľúča AES. Pamätajte, že to funguje vďaka prítomnosti S-Boxu v dátovom toku, na ktorý útočíme.*

*Ďalej uvidíme, ako namiesto konkrétnej hodnoty bitu použiť výkonovú analýzu. Pri tejto technike je cieľom rozdeliť priebehy spotreby podľa jedného bitu vo výsledku výstupu S-Boxu (nezáleží na tom, podľa ktorého): ak je tento bit 1, jeho skupina priebehov by mala mať v priemere vyššiu spotrebu energie počas operácie S-Boxu než druhá skupina.*

*To všetko je založené na predpoklade, ktorý sme preberali v prezentáciách a videli v predchádzajúcich laboch: existuje stály vzťah medzi hodnotou bitov na dátovej zbernici a spotrebou energie v zariadení.*

**CIELE VZDELÁVANIA:**

* Použitie merania spotreby na „overenie“ možného modelu zariadenia.
* Detekcia hodnoty jedného bitu pomocou merania spotreby.
* Prelomenie AES pomocou klasického útoku DPA.


## Požiadavky

Zadržte! Skôr než budete pokračovať, skontrolujte, či ste absolvovali nasledujúce tutoriály:

* ☑ Jupyter Notebook Intro (mali by ste byť v pohode s vykresľovaním grafov a spúšťaním blokov).
* ☑ SCA101 Intro (mali by ste mať predstavu o tom, ako spustiť verzie špecifické pre daný hardvér).
* ☑ Breaking AES Using a Single Bit (nadviažeme na vašu predchádzajúcu prácu).

## Model AES

Nemusíte si pamätať zložitý model z minulosti – namiesto toho môžeme skočiť priamo na model AES! Skopírujte nižšie svoj model AES, ktorý ste vytvorili v predchádzajúcom labe, a spustite ho:

In [ ]:
# ###################
# Add your code here
# ###################
#raise NotImplementedError("Add your code here, and delete this.")

# ###################
# START SOLUTION
# ###################
sbox = [
    # 0    1    2    3    4    5    6    7    8    9    a    b    c    d    e    f 
    0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76, # 0
    0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0, # 1
    0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15, # 2
    0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75, # 3
    0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84, # 4
    0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf, # 5
    0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8, # 6
    0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2, # 7
    0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73, # 8
    0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb, # 9
    0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79, # a
    0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08, # b
    0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a, # c
    0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e, # d
    0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf, # e
    0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16  # f
]

def aes_internal(inputdata, key):
    return sbox[inputdata ^ key]
# ###################
# END SOLUTION
# ###################

Správnosť modelu môžete overiť spustením nasledujúcich blokov, presne ako minule:

In [ ]:
#Simple test vectors - if you get the check-mark printed all OK.
assert(aes_internal(0xAB, 0xEF) == 0x1B)
assert(aes_internal(0x22, 0x01) == 0x26)
print("✔️ OK to continue!")

## AES Power Watcher (Sledovanie spotreby AES)

Ďalším krokom je posielanie náhodných dát do zariadenia a sledovanie spotreby energie počas šifrovania.

Myšlienka spočíva v tom, že použijeme zachytávaciu slučku (capture loop) takto:

    print(scope)
    for i in trange(N, desc='Capturing traces'):
        key, text = ktp.next()  # manual creation of a key, text pair can be substituted here

        trace = cw.capture_trace(scope, target, text, key)
        if trace is None:
            continue
        traces.append(trace)
        plot.send(trace)

    #Convert traces to numpy arrays
    trace_array = np.asarray([trace.wave for trace in traces])
    textin_array = np.asarray([trace.textin for trace in traces])
    known_keys = np.asarray([trace.key for trace in traces])  # for fixed key, these keys are all the same



In [ ]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWLITEXMEGA'
CRYPTO_TARGET='AVRCRYPTOLIB'
SS_VER='SS_VER_1_1'

In [ ]:
%run "../Setup_Scripts/Setup_Generic.ipynb"

In [ ]:
%%bash -s "$PLATFORM" "$CRYPTO_TARGET" "$SS_VER"
cd ../../firmware/mcu/simpleserial-aes
make PLATFORM=$1 CRYPTO_TARGET=$2 SS_VER=$3 -j

In [ ]:
cw.program_target(scope, prog, "../../firmware/mcu/simpleserial-aes/simpleserial-aes-{}.hex".format(PLATFORM))

In [ ]:
#
# Perform the capture, resulting in trace_array and textin_array of 2500 traces.
#
raise NotImplementedError("Add your code here, and delete this.")

assert(len(trace_array) == 2500)
print("✔️ OK to continue!")

<details>
<summary>Nejaké riešenie</summary>
<pre><code>
    from tqdm.notebook import trange
import numpy as np
import time

ktp = cw.ktp.Basic()
trace_array = []
textin_array = []

key, text = ktp.next()

target.set_key(key)

N = 2500
for i in trange(N, desc='Capturing traces'):
    scope.arm()
    
    target.simpleserial_write('p', text)
    
    ret = scope.capture()
    if ret:
        print("Target timed out!")
        continue
    
    response = target.simpleserial_read('r', 16)
    
    trace_array.append(scope.get_last_trace())
    textin_array.append(text)
    
    key, text = ktp.next() 

assert(len(trace_array) == 2500)
print("✔️ OK to continue!")
</code></pre>
</details>

Ako tieto dáta vyzerajú? Skúste si tu vykresliť jeden alebo dva priebehy:

In [ ]:
raise NotImplementedError("Add your code here, and delete this.")

<details>
<summary>Nejaké riešenie</summary>
<pre><code>
    %matplotlib widget
    import matplotlib.pyplot as plt
    import mplcursors

    plt.figure(figsize=(10,4))
    plt.plot(trace_array[0])
    plt.plot(trace_array[6])
    plt.show()
</code></pre>
</details>

Dobre, zaujímavé – takže máme dáta! A čo formát vstupných dát?

In [ ]:
print(textin_array[0])
print(textin_array[1])

## AES Guesser - Jeden bajt

Útok teraz potrebuje spôsob, ako rozdeliť priebehy do dvoch skupín v závislosti od stavu bitu v našej „tipovanej“ hodnote. Začneme zľahka tým, že budeme tipovať vždy jeden bajt AES kľúča.

Na začiatok definujte počet priebehov (traces) a počet bodov v každom priebehu. Môžete použiť nasledujúci ukážkový kód, stačí spustiť tento blok:

In [ ]:
numtraces = np.shape(trace_array)[0] #total number of traces
numpoints = np.shape(trace_array)[1] #samples per trace

Ak si pamätáte z prezentácií – náš algoritmus vyzerá takto:

    for key_byte_guess_value in [0, 1, 2, 3, ... 253, 254, 255]:
        
        one_list = prázdny zoznam
        zero_list = prázdny zoznam
        
        for trace_index in [0, 1, 2, 3, ..., numtraces]:
        
            input_byte = textin_array[trace_index][byte_to_attack]
            
            #Získajte zoznam hypotetického úniku (leakage) - použite aes_internal(guess, input_byte)          

            if bit 0 hypotetického úniku je 1:
                pridajte trace_array[trace_index] do one_list
            else:
                pridajte trace_array[trace_index] do zero_list
                
        one_avg = priemer z one_list
        zero_avg = priemer z zero_list

        max_diff_value = maximum z ABS(one_avg - zero_avg)
        
Na získanie priemeru vašich zoznamov `one_list` a `zero_list` môžete použiť numpy:

    import numpy as np
    avg_one_list = np.asarray(one_list).mean(axis=0)

Dôležitá vec je tu `axis=0`, čo spočíta priemer tak, že výsledné pole je vytvorené naprieč všetkými priebehmi (nie iba priemerná hodnota jedného priebehu, ale priemer každého indexu bodu *naprieč všetkými priebehmi*).

Aby som vám pomohol s testovaním – prezradím vám, že správna hodnota bajtu 0 je `0x2B`. Môžete to použiť na overenie, či vaše riešenie funguje na prvom bajte. Ak sa zaseknete – pozrite si tipy nižšie (ale najprv to skúste sami).

Mali by ste vidieť, že výstup maximálnej hodnoty rozdielu medzi dvoma priemernými skupinami bude vyšší pre hodnotu `0x2B`. Napríklad vypísanie maximálnej hodnoty SAD z ukážkového cyklu u mňa vyzerá takto:

    Guessing 28: 0.001397
    Guessing 29: 0.000927
    Guessing 2a: 0.001953
    Guessing 2b: 0.005278
    Guessing 2c: 0.000919
    Guessing 2d: 0.002510
    Guessing 2e: 0.001241
    Guessing 2f: 0.001242

Všimnite si hodnotu `0.005278` pre `0x2B` – táto je vyššia ako ostatné, ktoré sa pohybujú od `0.000927` do `0.002510`.

In [ ]:
# ###################
# Add your code here
# ###################
raise NotImplementedError("Add Your Code Here")

### Tip 1: Všeobecný tok programu

Na implementáciu vonkajšieho cyklu vyššie môžete použiť nasledujúci všeobecný tok programu:

In [ ]:
#Hint #1 - General Program Flow
import numpy as np
mean_diffs = np.zeros(256)

guessed_byte = 0

for guess in range(0, 256):
    
    one_list = []
    zero_list = []
    
    for trace_index in range(numtraces):
        #Inside here do the steps shown above
        pass
        
    #Do extra steps to average one_list and zero_list        

### Tip 2: Príklad dvoch rôznych tipov kľúča

Úplne vám to neprezradíme (ak to chcete, pozrite si notebook `SOLN`), ale tu je návod, ako môžete vygenerovať dva rozdiely, pre `0x2B` a `0xFF`. Ak ste úplne zaseknutí, môžete použiť nasledujúci kód ako základ pre to, čo by malo byť vo vnútri cyklov.

In [ ]:
import numpy as np
mean_diffs = np.zeros(256)

### Code to do guess of byte 0 set to 0x2B
guessed_byte = 0
guess = 0x2B
   
one_list = []
zero_list = []
    
for trace_index in range(numtraces):
    hypothetical_leakage = aes_internal(guess, textin_array[trace_index][guessed_byte])

    #Mask off the lowest bit - is it 0 or 1? Depending on that add trace to array
    if hypothetical_leakage & 0x01:        
        one_list.append(trace_array[trace_index])
    else:
        zero_list.append(trace_array[trace_index])
            
one_avg = np.asarray(one_list).mean(axis=0)
zero_avg = np.asarray(zero_list).mean(axis=0)
mean_diffs_2b = np.max(abs(one_avg - zero_avg))

print("Max SAD for 0x2B: {:1}".format(mean_diffs_2b))

### Code to do guess of byte 0 set to 0xFF
guessed_byte = 0
guess = 0xFF
    
one_list = []
zero_list = []
    
for trace_index in range(numtraces):
    hypothetical_leakage = aes_internal(guess, textin_array[trace_index][guessed_byte])

    #Mask off the lowest bit - is it 0 or 1? Depending on that add trace to array
    if hypothetical_leakage & 0x01:        
        one_list.append(trace_array[trace_index])
    else:
        zero_list.append(trace_array[trace_index])
            
one_avg = np.asarray(one_list).mean(axis=0)
zero_avg = np.asarray(zero_list).mean(axis=0)
mean_diffs_ff = np.max(abs(one_avg - zero_avg))

print("Max SAD for 0xFF: {:1}".format(mean_diffs_ff))

<details>
<summary>Nejaké riešenie</summary>
<pre><code>
    
for byte_to_attack in range(16):
    num_traces = len(trace_array)
    
    best_score = 0
    best_guess = None
    for key_byte_guess_value in range(256):
        
        one_list = []
        zero_list = []
        
        for trace_index in range(num_traces):
        
            input_byte = textin_array[trace_index][byte_to_attack]
            
            
            hypothetical_leakage_bit = (aes_internal(key_byte_guess_value,input_byte) >> 0) & 1
    
            if hypothetical_leakage_bit == 1:
                one_list.append(trace_array[trace_index])
            else:
                zero_list.append(trace_array[trace_index])
                
        one_avg = np.asarray(one_list).mean(axis=0)
        zero_avg = np.asarray(zero_list).mean(axis=0)
    
        max_diff = np.max(np.abs(one_avg - zero_avg))
        if max_diff > best_score:
            best_score = max_diff
            best_guess = key_byte_guess_value

    print(best_score)
    print(hex(best_guess))
</code></pre>
</details>

## Zoradenie tipov (Ranking Guesses)

Taktiež si pravdepodobne budete chcieť svoje tipy zoradiť (predpokladáme). To vám pomôže identifikovať najpravdepodobnejšiu hodnotu. Najlepší spôsob, ako to urobiť, je vytvoriť zoznam maximálnych hodnôt rozdielov pre každý kľúč:

    mean_diffs = [0]*256

    for key_byte_guess_value in [0, 1, 2, 3, ... 253, 254, 255]:

        *** PREDOSLI KOD ***
        max_diff_value = maximum z ABS(one_avg - zero_avg)
        mean_diffs[key_byte_guess_value] = max_diff_value
        
Ak upravíte svoj predchádzajúci kód, vygeneruje sa zoznam maximálnych rozdielov. Tento zoznam bude vyzerať ako:

    [0.002921, 0.001923, 0.005131, ..., 0.000984]
    
Kde *index* zoznamu je hodnota tipu kľúča. Môžeme použiť `np.argsort`, čo vygeneruje nový zoznam ukazujúci *indexy*, ktoré by zoradili pôvodný zoznam (o `argsort` ste sa mali dozvedieť aj v predchádzajúcom labe):

Takže napríklad spustite nasledujúci kód, aby ste ho videli v akcii na zozname `[1.0, 0.2, 3.4, 0.01]`:

In [ ]:
np.argsort([1.0, 0.2, 3.4, 0.01])

Toto by malo vrátiť `[3, 1, 0, 2]` – to je poradie od najnižšieho po najvyšší. Ak to chcete zmeniť od najvyššieho po najnižší, nezabudnite na koniec pridať `[::-1]`, napríklad `np.argsort([1.0, 0.2, 3.4, 0.01])[::-1]`.

Skúste použiť funkciu `np.argsort` na výstup najpravdepodobnejších hodnôt kľúča z vášho útoku.

## Vykresľovanie rozdielov

Skôr než sa posunieme ďalej – mali by ste sa pozrieť na rôzne grafy týchto rozdielov. Budú hrať rolu v niečom, čo sa nazýva problém *falošných špičiek* (ghost peak problem).

Teraz zadefinujeme funkciu s názvom `calculate_diffs()`, ktorá implementuje naše útoky (môžete ju nahradiť vlastnou funkciou alebo zatiaľ ponechať túto):

In [ ]:
def calculate_diffs(guess, byteindex=0, bitnum=0):
    """Perform a simple DPA on two traces, uses global `textin_array` and `trace_array` """
    
    one_list = []
    zero_list = []

    for trace_index in range(numtraces):
        hypothetical_leakage = aes_internal(guess, textin_array[trace_index][byteindex])

        #Mask off the requested bit
        if hypothetical_leakage & (1<<bitnum):
            one_list.append(trace_array[trace_index])
        else:
            zero_list.append(trace_array[trace_index])

    one_avg = np.asarray(one_list).mean(axis=0)
    zero_avg = np.asarray(zero_list).mean(axis=0)
    return abs(one_avg - zero_avg)

Skúste vykresliť rozdiel medzi rôznymi bajtmi. Pre bajt 0 si pamätajte, že `0x2B` je správna hodnota. Priblížte si grafy a všimnite si, ako by mal mať správny kľúč oveľa väčší rozdiel.

Niekedy dostávame *falošné špičky* (ghost peaks), čo sú nesprávne špičky. Zatiaľ predpokladáme, že existuje jediné „najlepšie“ riešenie pre kľúč – možno budeme musieť vymyslieť niečo sofistikovanejšie a nastaviť prah, čím získame niekoľko kandidátov na správny kľúč. Pre túto chvíľu si poďme vykresliť len pár príkladov:

In [ ]:
cw.plot(calculate_diffs(0x2B)) * cw.plot(calculate_diffs(0x2C)) * cw.plot(calculate_diffs(0x2D))

Tu je ukážka toho, ako by to malo vyzerať:

Všimnite si, že keď radíme bajty, používame iba maximálnu hodnotu akejkoľvek špičky. Z týchto grafov by ste sa mohli dozvedieť oveľa viac, napríklad polohu špičky alebo či je v grafe viacero špičiek. Nateraz však zostaneme pri

## AES Guesser - Všetky bajty

Výborne – dobrá práca! Máte hotový jeden bajt a zobrazené DPA grafy. Teraz sa posuňme vpred a skúsme uhádnuť *všetky* bajty.

To si vyžaduje trochu viac úsilia ako predtým. Vezmete svoju existujúcu funkciu na tipovanie a obalíte ju väčším cyklom zvonku, asi takto:

    for subkey in range(0,16):
        #Zvyšok kódu zpredtým!
        

In [ ]:
from tqdm.notebook import trange
import numpy as np

#Store your key_guess here, compare to known_key
key_guess = []
known_key = [0x2b, 0x7e, 0x15, 0x16, 0x28, 0xae, 0xd2, 0xa6, 0xab, 0xf7, 0x15, 0x88, 0x09, 0xcf, 0x4f, 0x3c]

for subkey in trange(0, 16, desc="Attacking Subkey"):
    # ###################
    # Add your code here
    # ###################
    raise NotImplementedError("Add Your Code Here")

<details>
<summary>Nejaké riešenie</summary>
<pre><code>
    from tqdm.notebook import trange
import numpy as np

#Store your key_guess here, compare to known_key
key_guess = []
known_key = [0x2b, 0x7e, 0x15, 0x16, 0x28, 0xae, 0xd2, 0xa6, 0xab, 0xf7, 0x15, 0x88, 0x09, 0xcf, 0x4f, 0x3c]


for byte_to_attack in range(16):
    num_traces = len(trace_array)
    
    best_score = 0
    best_guess = None
    for key_byte_guess_value in range(256):
        
        one_list = []
        zero_list = []
        
        for trace_index in range(num_traces):
        
            input_byte = textin_array[trace_index][byte_to_attack]
            
            
            hypothetical_leakage_bit = (aes_internal(key_byte_guess_value,input_byte) >> 0) & 1
    
            if hypothetical_leakage_bit == 1:
                one_list.append(trace_array[trace_index])
            else:
                zero_list.append(trace_array[trace_index])
                
        one_avg = np.asarray(one_list).mean(axis=0)
        zero_avg = np.asarray(zero_list).mean(axis=0)
    
        max_diff = np.max(np.abs(one_avg - zero_avg))
        if max_diff > best_score:
            best_score = max_diff
            best_guess = key_byte_guess_value

    key_guess.append(best_guess) 
print(key_guess)
print(known_key)
</code></pre>
</details>

🥳🥳🥳🥳🥳🥳🥳🥳🥳🥳🥳🥳🥳
Gratulujem – dokázali ste to!!!!

Dúfajme, že vyššie uvedený kód fungoval – ale pôjdeme ešte o kúsok ďalej, aby sme pochopili, ako to aplikovať v prípade, že to nefungovalo hneď na prvýkrát (alebo to fungovalo takmer správne).

## Falošné špičky (Ghost Peaks)

Možno sa v predchádzajúcom kroku nepodarilo obnoviť úplný kľúč? Netreba sa obávať – existuje pre to niekoľko dôvodov. Jedným z artefaktov DPA útoku je, že dostanete ďalšiu silnú špičku, ktorá ale nie je správnym kľúčom (čo môže byť práve falošná špička).

K efektívnejším útokom sa dostaneme neskôr, ale nateraz sa pozrime na niektoré riešenia:

* Zvýšte počet zaznamenaných priebehov (traces).
* Zmeňte cieľový bit (a skombinujte riešenia z viacerých bitov).
* Použite orezanie (windowing) vstupných dát.

Prvá možnosť je riešenie hrubou silou: prejdite z 2500 na 5000 alebo dokonca 10000 priebehov spotreby. Ako budete pridávať viac dát, môžete zistiť, že sa problém zmenšuje. Skutočné falošné špičky však nemusia zmiznúť, takže musíme prejsť k iným riešeniam.

Skôr než začneme – dáme vám „overene funkčný“ skript pre DPA útok, na ktorom budeme stavať. Tento skript využíva funkciu `calculate_diffs()` definovanú skôr.

Spustite nasledujúci blok (bude to chvíľu trvať):

In [ ]:
from tqdm.notebook import trange
import numpy as np

#Store your key_guess here, compare to known_key
key_guess = []
known_key = [0x2b, 0x7e, 0x15, 0x16, 0x28, 0xae, 0xd2, 0xa6, 0xab, 0xf7, 0x15, 0x88, 0x09, 0xcf, 0x4f, 0x3c]

#Which bit to target
bitnum = 0

full_diffs_list = []

for subkey in trange(0, 1, desc="Attacking Subkey"):
    
    max_diffs = [0]*256
    full_diffs = [0]*256

    for guess in range(0, 256):
        full_diff_trace = calculate_diffs(guess, subkey, bitnum)
        max_diffs[guess] = np.max(full_diff_trace)
        full_diffs[guess] = full_diff_trace
    
    #Make copy of the list
    full_diffs_list.append(full_diffs[:])
    
    #Get argument sort, as each index is the actual key guess.
    sorted_args = np.argsort(max_diffs)[::-1]
    
    #Keep most likely
    key_guess.append(sorted_args[0])
    
    #Print results
    print("Subkey %2d - most likely %02X (actual %02X)"%(subkey, key_guess[subkey], known_key[subkey]))
    
    #Print other top guesses
    print(" Top 5 guesses: ")
    for i in range(0, 5):
        g = sorted_args[i]
        print("   %02X - Diff = %f"%(g, max_diffs[g]))
    
    print("\n")

Tento blok by mal teraz vypísať niekoľko *ďalších najlepších tipov* – v tomto prípade len top 5 nasledujúcich tipov, ale ak si želáte, môžete to rozšíriť. Taktiež si uchováva kópiu všetkých *rozdielových* priebehov (na rozdiel od minulosti, kedy ich zahadzoval).

### Vykresľovanie špičiek

Po spustení vyberte subkľúč, ktorý je buď nesprávny, alebo má veľmi blízke „ďalšie najlepšie tipy“. Nasledujúci príklad ukazuje výstup, kde je Subkey 5 vlastne nesprávny – správny tip (`0xAE`) bol zaradený až na 3. miesto (option 3 v zozname).

    Subkey  5 - most likely CB (actual AE)
     Top 5 guesses: 
       CB - Diff = 0.003006
       C5 - Diff = 0.002984
       AE - Diff = 0.002739
       3C - Diff = 0.002674
       2F - Diff = 0.002511

Úplný rozdiel nájdete v poli `full_diffs_list`. Ak toto pole indexujete, poskytne vám každý tip pre daný subkľúč (napríklad `full_diffs_list[5]` sú výstupy tipov pre 5. subkľúč).

Použitím `full_diffs_list[N]` pre váš vybraný subkľúč vykreslite správny kľúč tak, že vykreslíte `full_diffs_list[N][0xCORRECT]` zelenou farbou ako *posledný* (aby sa zobrazil navrchu). Predtým vykreslite niekoľko iných vysoko umiestnených tipov. V mojom príklade by to vyzeralo takto:

    %matplotlib notebook
    import matplotlib.pylab as plt

    plt.plot(full_diffs_list[5][0xC5], 'r')
    plt.plot(full_diffs_list[5][0xCB], 'r')
    plt.plot(full_diffs_list[5][0xAE], 'g')

In [ ]:
cw.plot(full_diffs_list[5][0xC5]) * cw.plot(full_diffs_list[5][0xCB]) * cw.plot(full_diffs_list[5][0xAE])

Priblížte si okno a mali by ste si všimnúť, že existuje miesto, kde je správna špička *vyššia* než tie nesprávne. Ak by ste chceli vykresliť všetky priebehy pre daný subkľúč (bude to pomalé!), môžeme to urobiť nasledovne:

In [ ]:
fig = cw.plot()
subkey = 12
for guess in range(0, 256):
    fig *= cw.plot(full_diffs_list[subkey][guess])
fig

V závislosti od vášho hardvéru môže predchádzajúci krok ukázať jednu peknú veľkú špičku alebo viacero veľkých špičiek. Ak máme problém s falošnými špičkami, pravdepodobne máte špičiek viacero. Nesprávne špičky môžu zaostávať za správnymi miestami – správne miesta môžeme najprv vykresliť tak, že sa pozrieme na známy kľúč. Nasledujúci kód to urobí:

In [ ]:
fig = cw.plot()
for subkey in range(0, 16):
    fig *= cw.plot(full_diffs_list[subkey][known_key[subkey]])
fig

### Orezanie špičiek (Windowing Peaks)

Posledný trik – zistite, či existuje nejaký spôsob, ako „orezať“ (window) dáta tak, aby to bolo užitočné. Napríklad pri pohľade na špičky si môžete všimnúť, že správne špičky prichádzajú vždy s posunom 60 cyklov, pričom prvá špička je okolo vzorky 1100 (tieto hodnoty sa budú pre váš hardvér líšiť).

Takže by sme mohli upraviť cyklus tak, aby hľadal rozdiely až po tomto bode:

    for guess in range(0, 256):
        full_diff_trace = calculate_diffs(guess, subkey, bitnum)
        full_diff_trace = full_diff_trace[(1010 + subkey*60):]
        max_diffs[guess] = np.max(full_diff_trace)
        full_diffs[guess] = full_diff_trace
        
Skopírujte sem celý DPA útok – a vyskúšajte ho! Uvidíte, či sa vám podarí dostať správny kľúč pre každý jeden bajt.

In [ ]:
# ###################
# Add your code here
# ###################
raise NotImplementedError("Add Your Code Here")

### Útok na ostatné bity

Doteraz sme sa pozerali iba na bit 0 – ale v hre je viac bitov! Najprv môžete jednoducho vyskúšať iný bit, ktorý by mohol byť prítomný, možno budú fungovať lepšie?

Môžete však tiež skombinovať viacero bitov vytvorením najpravdepodobnejšieho riešenia, ktoré sa aplikuje naprieč *všetkými* bitmi.

Prvú možnosť je jednoduché vyskúšať, keďže útočený bit sme si zadefinovali v predchádzajúcom skripte.

Druhá možnosť je o niečo pokročilejšia. Môžete ju vyskúšať – ale v praxi sa na efektívne kombinovanie viacerých bitov používa metóda nazývaná CPA útok.

## Závery a ďalšie kroky

Teraz ste videli, ako sa dá vykonať DPA útok pomocou jednoduchého Python skriptu. Oveľa efektívnejšie útoky zažijeme, keď sa pozrieme na CPA útok.

Ak chcete tieto útoky vykonávať v praxi, Python kód tu nie je najefektívnejší! Na rýchlejšie možnosti sa pozrieme v neskorších kurzoch.

In [ ]:
target.dis()
scope.dis()

---
<small>UPOZORNENIE: Tento materiál je chránený autorskými právami (C) NewAE Technology Inc., 2015–2020. ChipWhisperer je ochranná známka spoločnosti NewAE Technology Inc., na ktorú si táto spoločnosť uplatňuje práva vo všetkých jurisdikciách a ktorá je registrovaná minimálne v Spojených štátoch amerických, Európskej únii a Čínskej ľudovej republike.

Návody odvodené z našej open-source práce musia byť uverejnené pod príslušnou open-source licenciou a upozornenie na zdroj musí byť *jasne zobrazené*. Iba pôvodní držitelia autorských práv môžu udeľovať licencie alebo povoľovať ďalšiu distribúciu – hoci spoločnosť NewAE Technology Inc. vlastní autorské práva k mnohým návodom, repozitár github obsahuje príspevky komunity, ktoré nemôže licencovať za špeciálnych podmienok a **musia** byť udržiavané ako open-source verzia. V prípade potreby prosím kontaktujte NewAE Technology Inc ohľadom špeciálnych povolení (ak je to možné)

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.</small>